# Word-count map-reduce

Generate a synthetic corpus on the client, upload it once with ``send_object``, then fan out chunk-offset tasks that tokenize their slice and return a small ``Counter``. The client merges the per-chunk counters and prints the top words.

This is the textbook map-reduce pattern: a single shared input, many independent map tasks each emitting a partial result, and a cheap reduce on the client. The corpus is sent across the wire exactly once thanks to ``send_object``.

**Prerequisites**: a Scaler scheduler reachable from this kernel. See :doc:`../tutorials/try_in_browser` for the recommended worker setup.

In [ ]:
# Connection settings -- edit these to point at your running cluster.
SCHEDULER_ADDRESS = "ws://127.0.0.1:2345"  # supports tcp:// or ws://; only ws:// works from JupyterLite (browser)
OBJECT_STORAGE_ADDRESS = None  # leave None to use whatever the scheduler advertises

N_CHUNKS = 32
CHUNK_SIZE_BYTES = 256 * 1024  # 256 KiB per chunk -> 8 MiB total corpus

In [ ]:
# Build a deterministic synthetic corpus on the client. Kept small enough that
# we never block the browser kernel: 8 MiB of plain ASCII for the default knobs.
import random
import time

from collections import Counter

from scaler import Client

VOCAB = (
    "lorem ipsum dolor sit amet consectetur adipiscing elit sed do eiusmod"
    " tempor incididunt ut labore et dolore magna aliqua scaler distributed"
    " computing python parfun pargraph cluster scheduler worker task future"
    " submit object storage map reduce parallel"
).split()

rng = random.Random(0)
corpus_size = N_CHUNKS * CHUNK_SIZE_BYTES
pieces: list[str] = []
running = 0
while running < corpus_size:
    word = rng.choice(VOCAB)
    pieces.append(word)
    running += len(word) + 1
corpus = " ".join(pieces)[:corpus_size]
print(f"generated {len(corpus):,} bytes of corpus across {len(set(VOCAB))} unique vocab words")

In [ ]:
def count_words_in_chunk(corpus: str, start: int, end: int) -> Counter:
    """Worker-side: tokenize a slice of the corpus and return a Counter."""
    # Expand the slice to the nearest whitespace boundaries so we never split a word.
    if start > 0:
        while start < len(corpus) and not corpus[start].isspace():
            start += 1
    if end < len(corpus):
        while end < len(corpus) and not corpus[end].isspace():
            end += 1
    return Counter(corpus[start:end].split())


chunk_bounds = [(i * CHUNK_SIZE_BYTES, (i + 1) * CHUNK_SIZE_BYTES) for i in range(N_CHUNKS)]

with Client(address=SCHEDULER_ADDRESS, object_storage_address=OBJECT_STORAGE_ADDRESS) as client:
    corpus_ref = client.send_object(corpus, name="corpus")
    started = time.perf_counter()
    futures = [client.submit(count_words_in_chunk, corpus_ref, lo, hi) for lo, hi in chunk_bounds]
    totals: Counter = Counter()
    for future in futures:
        totals.update(future.result())
    elapsed = time.perf_counter() - started

print(f"map-reduced {sum(totals.values()):,} tokens across {N_CHUNKS} chunks in {elapsed:.2f}s")
for word, count in totals.most_common(10):
    print(f"  {word:<14} {count:>8,}")